##Taining

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving uniprotkb_AND_reviewed_true_AND_model_o_2026_03_23.fasta.gz to uniprotkb_AND_reviewed_true_AND_model_o_2026_03_23.fasta.gz


In [ ]:
!gunzip /content/uniprotkb_AND_reviewed_true_AND_model_o_2026_03_23.fasta.gz

In [ ]:
from Bio import SeqIO

file_name = "/content/uniprotkb_AND_reviewed_true_AND_model_o_2026_03_23.fasta"

sequences = []

for record in SeqIO.parse(file_name, "fasta"):
    seq = str(record.seq)
    if "*" not in seq and "X" not in seq:
        sequences.append(seq)

print("Total sequences:", len(sequences))
print("Example:", sequences[0])

Total sequences: 20431
Example: MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNLLHVDFQNTPYCFDQLRRRFGDVFSLQLAWTPVVVLNGLAAVREAMVTRGEDTADRPPAPIYQVLGFGPRSQGVILSRYGPAWREQRRFSVSTLRNLGLGKKSLEQWVTEEAACLCAAFADQAGRPFRPNGLLDKAVSNVIASLTCGRRFEYDDPRFLRLLDLAQEGLKEESGFLREVLNAVPVLPHIPALAGKVLRFQKAFLTQLDELLTEHRMTWDPAQPPRDLTEAFLAKKEKAKGSPESSFNDENLRIVVGNLFLAGMVTTSTTLAWGLLLMILHLDVQRGRRVSPGCPIVGTHVCPVRVQQEIDDVIGQVRRPEMGDQAHMPCTTAVIHEVQHFGDIVPLGVTHMTSRDIEVQGFRIPKGTTLITNLSSVLKDEAVWKKPFRFHPEHFLDAQGHFVKPEAFLPFSAGRRACLGEPLARMELFLFFTSLLQHFSFSVAAGQPRPSHSRVVSFLVTPSPYELCAVPR


In [ ]:
MAX_LEN = 100

processed_sequences = []

for seq in sequences:
    if len(seq) >= MAX_LEN:
        processed_sequences.append(seq[:MAX_LEN])

print("Processed:", len(processed_sequences))

Processed: 19676


In [ ]:
# Encoding step

chars = sorted(list(set(''.join(processed_sequences))))

char_to_int = {c:i for i,c in enumerate(chars)}
int_to_char = {i:c for c,i in char_to_int.items()}

vocab_size = len(chars)

print("Vocab size:", vocab_size)

Vocab size: 21


In [ ]:
seq_length = 30

X = []
y = []

for seq in processed_sequences:
    for i in range(len(seq) - seq_length):
        input_seq = seq[i:i+seq_length]
        output_char = seq[i+seq_length]

        X.append([char_to_int[c] for c in input_seq])
        y.append(char_to_int[output_char])

import numpy as np
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1377320, 30)
y shape: (1377320,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Embedding

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    GRU(128),
    Dense(vocab_size, activation='softmax')
])

# Build model manually (THIS IS THE FIX)
model.build(input_shape=(None, seq_length))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 64)         │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 21)             │         2,709 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,549 (306.83 KB)

 Trainable params: 78,549 (306.83 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Stop training if loss doesn't improve for 5 epochs
early_stop = EarlyStopping(
    monitor='loss',
    patience=5,           # wait 5 epochs before stopping
    restore_best_weights=True,
    verbose=1
)

# Save best model automatically during training
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='loss',
    save_best_only=True,  # only saves when loss improves
    verbose=1
)

# Now train — it will stop automatically when ready
history = model.fit(
    X, y,
    epochs=100,           # max epochs — won't necessarily reach 100
    batch_size=256,       # larger batch = faster on GPU (was 64 before)
    callbacks=[early_stop, checkpoint]
)

Epoch 1/100
5374/5381 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.8561
Epoch 1: loss improved from None to 2.84085, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
5381/5381 ━━━━━━━━━━━━━━━━━━━━ 34s 6ms/step - loss: 2.8408
Epoch 2/100
5376/5381 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.8200
Epoch 2: loss improved from 2.84085 to 2.81483, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
5381/5381 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 2.8148
Epoch 3/100
5376/5381 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.8014
Epoch 3: loss improved from 2.81483 to 2.79837, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
5381/5381 ━━━━━━━━━━━━━━━━━━━━ 30s 6ms/step - loss: 2.7984
Epoch 4/100
5378/5381 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.7876
Epoch 4: loss improved from 2.79837 to 2.78511, saving model to best_model.keras

Epoch 4: finished saving model to best_model.keras
5381/5381 ━━━━

##Run

In [1]:
!pip install biopython

import numpy as np
import random
import tensorflow as tf
from tensorflow import keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.6 MB/s eta 0:00:00


In [2]:
!wget --content-disposition "https://drive.usercontent.google.com/u/0/uc?id=1up2745OHmGaG2qlooaBifrN36cy9_7Kq&export=download"
!wget --content-disposition "https://drive.usercontent.google.com/u/0/uc?id=1AVTt4Gk_6FnT15J_OIJ5pbCiLpMnJfaP&export=download"

--2026-04-12 09:15:27--  https://drive.usercontent.google.com/u/0/uc?id=1up2745OHmGaG2qlooaBifrN36cy9_7Kq&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.141.132, 2607:f8b0:400c:c06::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.141.132|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://drive.usercontent.google.com/uc?id=1up2745OHmGaG2qlooaBifrN36cy9_7Kq&export=download [following]
--2026-04-12 09:15:28--  https://drive.usercontent.google.com/uc?id=1up2745OHmGaG2qlooaBifrN36cy9_7Kq&export=download
Reusing existing connection to drive.usercontent.google.com:443.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1up2745OHmGaG2qlooaBifrN36cy9_7Kq&export=download [following]
--2026-04-12 09:15:28--  https://drive.usercontent.google.com/download?id=1up2745OHmGaG2qlooaBifrN36cy9_7Kq&export=download
Reusing exi

In [3]:
dataset_filename="uniprotkb_proteome_UP000005640_2026_04_12.fasta"
model_filename="model.keras"

In [4]:
from Bio import SeqIO

processed_sequences = []

try:
    for record in SeqIO.parse(dataset_filename, "fasta"):
        seq = str(record.seq).upper()
        if all(c in "ACDEFGHIKLMNPQRSTVWY" for c in seq):
            processed_sequences.append(seq)
except Exception:
    with open(dataset_filename) as f:
        for line in f:
            seq = line.strip().upper()
            if seq and all(c in "ACDEFGHIKLMNPQRSTVWY" for c in seq):
                processed_sequences.append(seq)

print(f"Loaded {len(processed_sequences)} sequences")

# Build char mappings — match model's actual output size
amino_acids = sorted(set("".join(processed_sequences)))
char_to_int = {c: i for i, c in enumerate(amino_acids)}
int_to_char = {i: c for c, i in char_to_int.items()}
vocab_size = len(char_to_int)

# Load model first to check output size
model = keras.models.load_model(model_filename)
model_vocab_size = model.output_shape[-1]

# If model has extra token (e.g. padding), add it to int_to_char
if model_vocab_size > vocab_size:
    for i in range(vocab_size, model_vocab_size):
        int_to_char[i] = None  # mark unknown tokens explicitly

print(f"Vocab size: {vocab_size}, AAs: {''.join(amino_acids)}")
print(f"Model output size: {model_vocab_size}")
model.summary()

Loaded 77024 sequences
Vocab size: 20, AAs: ACDEFGHIKLMNPQRSTVWY
Model output size: 21


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 64)         │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 21)             │         2,709 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,649 (920.51 KB)

 Trainable params: 78,549 (306.83 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 157,100 (613.68 KB)

In [5]:
def generate_sequence(model, seed, length=100, temperature=1.0):
    seq = list(seed)
    seq_len = model.input_shape[1]
    model_vocab_size = model.output_shape[-1]

    for _ in range(length):
        x = [char_to_int.get(c, 0) for c in seq[-seq_len:]]
        x = np.array(x).reshape(1, -1)

        preds = model.predict(x, verbose=0)[0]
        preds = np.asarray(preds).astype("float64")
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))

        preds[model_vocab_size - 1] = 0.0
        preds = preds / preds.sum()

        next_idx = np.random.choice(len(preds), p=preds)
        next_char = int_to_char.get(next_idx, None)
        if next_char is None:
            continue
        seq.append(next_char)

    return "".join(seq)

seed = random.choice(processed_sequences)[:30]

In [6]:
def score_protein(seq):
    score = 0
    length = len(seq)
    if 95 <= length <= 105:
        score += 5
    elif 80 <= length <= 120:
        score += 2
    else:
        score -= 2
    for aa in set(seq):
        if seq.count(aa) / length > 0.3:
            score -= 3
    diversity = len(set(seq)) / length
    score += diversity * 10
    hydrophobic = "AILMFWV"
    hydro_ratio = sum(seq.count(c) for c in hydrophobic) / length
    score += (1 - abs(hydro_ratio - 0.45)) * 5
    charged = "KRDE"
    charge_ratio = sum(seq.count(c) for c in charged) / length
    if 0.1 <= charge_ratio <= 0.3:
        score += 2
    else:
        score -= 1
    bad_patterns = ["LLLLLL", "AAAAAA", "VVVVVV"]
    for pattern in bad_patterns:
        if pattern in seq:
            score -= 3
    return round(score, 3)

def is_valid_protein(seq):
    return "*" not in seq and len(seq) >= 30 and "LLLLLLLL" not in seq

def mutate_seed(seed, mutation_rate=0.1):
    seed = list(seed)
    for i in range(len(seed)):
        if random.random() < mutation_rate:
            seed[i] = random.choice(list(char_to_int.keys()))
    return "".join(seed)

# Similarity functions
def similarity(seq1, seq2):
    matches = sum(c1 == c2 for c1, c2 in zip(seq1, seq2))
    return matches / len(seq1)

def check_novelty(generated_seq, dataset):
    max_sim = 0
    for seq in dataset:
        sim = similarity(generated_seq[:len(seq)], seq[:len(generated_seq)])
        if sim > max_sim:
            max_sim = sim
    return max_sim

# Generate 10 sequences and rank them
generated_list = []
for _ in range(10):
    base_seed = random.choice(processed_sequences)[:30]
    seed = mutate_seed(base_seed)
    seq = generate_sequence(model, seed, 100, temperature=1.2)
    generated_list.append(seq)

results = [(seq, score_protein(seq)) for seq in generated_list if is_valid_protein(seq)]
results = sorted(results, key=lambda x: x[1], reverse=True)

# Print top 5 with score AND similarity
print(f"\nTop {min(5, len(results))} proteins:\n")
for seq, score in results[:5]:
    sim = check_novelty(seq, processed_sequences[:100])
    print("Protein")
    print("Score:", score)
    print("Similarity:", round(sim, 3))
    print(seq)
    print()


Top 5 proteins:

Protein
Score: 6.442
Similarity: 0.125
MAFVPVIPESPSHVLAEFESLDPLLSALFHTLPCWFFCIAAGWFYNIEHLIWRSFRIQTTQISIWFPQQLKMEHATWSLLLKTSHFGAINSLPWTAGLGALLSEDAGREAFQALCNEKAHGWLWPTRPET

Protein
Score: 6.404
Similarity: 0.131
MAQLFLPLLAALVLAQAPAALARVLEGDSSWLWGCPAARAPEPGERGSEWEGGAWDCIYYTNTLHFQTDRPWLRLLKPLRSSLDECLYLPLEQTSPGWLLRASEENLQDIAWWSFRKDLPLQWSCYGINP

Protein
Score: 6.288
Similarity: 0.188
MVFSSNDEGLINKKLPKELLNRMLGELLLNWSQHPYMLPNEMHAKASAMGEFPFHLLKIGLRPSQSQGNGAAYGAAHWPWTAMGLLRRFSCHISPEWLSQWESLLFRRFLLESGIGDEQSLRQEPARAEG

Protein
Score: 6.173
Similarity: 0.136
MTSSPVTGGDETDCGPSLGLAAGIVLLVATIPLLLADLASMLGKTLEPELFSQRPYKLLGKFGPWQTFNWPHLFASNWLLESNLRMSLEDRNPQMEYPLIKAIHQKNKWKMQPEGWDKQNKEITRKHWDD

Protein
Score: 6.173
Similarity: 0.125
MCLEMLNPIHHYITSIVPEAMPAATMPVKLGTLFWWQDPWRPIEPKMALPPWCIAEPKMKDNSLWYKGELDLQWRREMTPRLEESTQRLPGTRSGELLDRSKAKQCFQQWLRSAFDGWAWLKNRKEDFRP



In [7]:
!pip install py3Dmol


In [8]:
import requests
import py3Dmol
for i, (seq, score) in enumerate(results[:5]):
    print(f"\n--- Protein #{i+1} | Score: {score} ---")
    print(seq)

    resp = requests.post(
        "https://api.esmatlas.com/foldSequence/v1/pdb/",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data=seq
    )

    view = py3Dmol.view(width=700, height=400)
    view.addModel(resp.text, "pdb")
    view.setStyle({"cartoon": {"colorscheme": "ssJmol"}})
    view.addSurface(py3Dmol.SAS, {"opacity": 0.12})
    view.zoomTo()
    view.spin(False)
    view.show()


--- Protein #1 | Score: 6.442 ---
MAFVPVIPESPSHVLAEFESLDPLLSALFHTLPCWFFCIAAGWFYNIEHLIWRSFRIQTTQISIWFPQQLKMEHATWSLLLKTSHFGAINSLPWTAGLGALLSEDAGREAFQALCNEKAHGWLWPTRPET


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Protein #2 | Score: 6.404 ---
MAQLFLPLLAALVLAQAPAALARVLEGDSSWLWGCPAARAPEPGERGSEWEGGAWDCIYYTNTLHFQTDRPWLRLLKPLRSSLDECLYLPLEQTSPGWLLRASEENLQDIAWWSFRKDLPLQWSCYGINP


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Protein #3 | Score: 6.288 ---
MVFSSNDEGLINKKLPKELLNRMLGELLLNWSQHPYMLPNEMHAKASAMGEFPFHLLKIGLRPSQSQGNGAAYGAAHWPWTAMGLLRRFSCHISPEWLSQWESLLFRRFLLESGIGDEQSLRQEPARAEG


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Protein #4 | Score: 6.173 ---
MTSSPVTGGDETDCGPSLGLAAGIVLLVATIPLLLADLASMLGKTLEPELFSQRPYKLLGKFGPWQTFNWPHLFASNWLLESNLRMSLEDRNPQMEYPLIKAIHQKNKWKMQPEGWDKQNKEITRKHWDD


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Protein #5 | Score: 6.173 ---
MCLEMLNPIHHYITSIVPEAMPAATMPVKLGTLFWWQDPWRPIEPKMALPPWCIAEPKMKDNSLWYKGELDLQWRREMTPRLEESTQRLPGTRSGELLDRSKAKQCFQQWLRSAFDGWAWLKNRKEDFRP


3Dmol.js failed to load for some reason. Please check your browser console for error messages.